<a href="https://colab.research.google.com/github/ArtMart-kn/Lab-8_Web-technology_Martynovskiy-9-/blob/main/LR01_setup_Martynovskiy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота 1. Налаштування робочого середовища

**Дисципліна:** Інженерія даних з Python  •  **Обсяг:** 2 год  •  **Тема 1**  •  **Версія 2**

Кафедра комп'ютерних наук, Кам'янець-Подільський національний університет імені Івана Огієнка

---

**Виконав(ла):**

Студент(ка) групи: **KN1-B23**

Прізвище, ім'я: **Мартиновський Артьом**

Посилання на блокнот:___________________
---

**Домашня підготовка:** мікрокурс [Python](https://www.kaggle.com/learn/python) платформи Kaggle Learn, завдання 1–7.

**Опорний конспект теми:** `T01_intro_pandas.ipynb`

> **Перш ніж почати:** оберіть *Файл → Зберегти копію на Диску*.

## Мета роботи

Налаштувати середовище, у якому виконуватимуться всі наступні лабораторні роботи семестру, і переконатися, що всі канали доступу до даних працюють.

**Після виконання ви матимете:**

1. робочу теку на Google Диску;
2. налаштований доступ до Kaggle через програмний інтерфейс, з ключем у секретах Colab;
3. налаштований доступ до Google BigQuery через режим Sandbox;
4. власний репозиторій GitHub із збереженим блокнотом.

**Найдовший крок** — створення проєкту в Google Cloud. Зробіть його вдома напередодні заняття, тоді на парі залишиться саме робота, а не очікування.

> **Важливо:** усю роботу виконуйте під **власним** обліковим записом Google. Секрети Colab прив'язані до акаунта, тому під спільним акаунтом лабораторії налаштування не збережеться.

## Крок 0. Перевірка середовища

Перша комірка кожного блокнота курсу друкує версії бібліотек. Це не формальність: Colab оновлюється без попередження, і якщо код колись перестане працювати, ви одразу побачите, що змінилося.

In [2]:
import sys
import numpy as np
import pandas as pd

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy :", np.__version__)

pd.set_option("display.max_rows", 12)
pd.set_option("display.width", 120)

Python: 3.13.18
pandas: 2.2.3
numpy : 2.4.6


## Крок 1. Google Диск

Обчислювальна сесія Colab тимчасова: після її зупинки всі створені файли зникають. Тому результати роботи зберігаємо на Диск.

Виконайте комірку — Colab запитає дозвіл.

In [1]:
from pathlib import Path

DRIVE_AVAILABLE = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_AVAILABLE = Path("/content/drive/MyDrive").is_dir()
    print("Google Диск підключено:", DRIVE_AVAILABLE)
except Exception as error:
    print("Google Диск не підключено:", error)
    print("Робота продовжиться у тимчасовій теці Colab. Для постійного збереження запустіть блокнот у Google Colab і надайте доступ до Диска.")

Mounted at /content/drive
Google Диск підключено: True


In [3]:
from pathlib import Path

if DRIVE_AVAILABLE:
    РОБОЧА = Path("/content/drive/MyDrive/data-engineering/lab01")
else:
    РОБОЧА = Path("/content/data-engineering/lab01")

РОБОЧА.mkdir(parents=True, exist_ok=True)
print("Робоча тека:", РОБОЧА)
print("Існує:", РОБОЧА.exists())

Робоча тека: /content/drive/MyDrive/data-engineering/lab01
Існує: True


### Завдання 1.1

Запишіть у робочу теку файл і прочитайте його назад. Це найпростіша перевірка того, що Диск підключений на запис, а не лише на читання.

Створіть таблицю з **трьох рядків**, у якій будуть ваше прізвище, номер за списком групи і поточна дата. Збережіть її у файл `РОБОЧА / 'perevirka.csv'` і прочитайте назад у змінну `perevirka`.

> Не забудьте `index=False` при записі — інакше при читанні зʼявиться зайвий стовпець `Unnamed: 0`.

In [4]:
from datetime import date

НОМЕР_ЗА_СПИСКОМ = "10"

perevirka = pd.DataFrame({
    "Прізвище": ["Мартиновський"] * 3,
    "Номер за списком": [НОМЕР_ЗА_СПИСКОМ] * 3,
    "Дата": [date.today().isoformat()] * 3
})

шлях_csv = РОБОЧА / "perevirka.csv"
perevirka.to_csv(шлях_csv, index=False, encoding="utf-8-sig")
perevirka = pd.read_csv(шлях_csv)
print("CSV успішно записано та прочитано:", шлях_csv)
display(perevirka)

CSV успішно записано та прочитано: /content/drive/MyDrive/data-engineering/lab01/perevirka.csv


,Прізвище,Номер за списком,Дата
0,Мартиновський,10,2026-11-08
1,Мартиновський,10,2026-11-08
2,Мартиновський,10,2026-11-08


## Крок 2. Kaggle API

Програмний інтерфейс Kaggle дозволяє завантажувати набори даних командою, а не через браузер. Це важливо для відтворюваності: команда в блокноті документує, звідки взялися дані, а завантаження мишею — ні.

Доступ до API потребує двох значень — імені користувача й ключа. Зберігати їх ми будемо в **секретах Colab**, а не в коді блокнота.

### 2.1. Отримання облікових даних

1. Відкрийте kaggle.com → ваш профіль → *Settings* → вкладка *API Tokens*.
2. На сторінці **дві секції**. Потрібна **нижня** — **Legacy API Credentials**. Натисніть у ній кнопку **`Create Legacy API Key`**.

> **Верхня секція не підходить.** Вона називається *API Tokens (Recommended)* і має кнопку `Generate New Token`. Попри слово «Recommended», цей новий формат наш код не приймає: він працює з парою «ім'я користувача + ключ», а новий токен — це одне значення іншого вигляду. Тиснемо саме `Create Legacy API Key`.

3. Завантажиться файл `kaggle.json`. Окремої кнопки «показати токен» на Kaggle немає — значення лежать усередині цього файлу.
4. Відкрийте `kaggle.json` **будь-яким текстовим редактором**. Усередині два поля:

```
{"username":"ivan_student","key":"3f8a2b9c1d4e..."}
```

5. Ці два значення знадобляться на наступному кроці. Нікуди їх не пересилайте — вони підуть одразу в секрети Colab.

### 2.2. Збереження в секретах Colab

Секрет складається з двох частин: **назви** і **значення**. Назва — це ярлик, за яким код знаходить секрет; вона однакова в усіх і **вже прописана в коді нижче**. Значення — ваше особисте, з файлу `kaggle.json`.

| Поле «Назва» | Поле «Значення» |
|---|---|
| `KAGGLE_USERNAME` | ваш нікнейм — те, що у файлі стоїть після `"username":` |
| `KAGGLE_KEY` | довгий ключ — те, що у файлі стоїть після `"key":` |

1. На лівій панелі Colab натисніть іконку **ключа** (*Secrets* / *Секрети*), далі **«+ Додати секрет»**.
2. Заповніть перший рядок за таблицею вище, потім додайте другий секрет і заповніть його.
3. Для обох увімкніть перемикач **«Доступ для блокнота»** — він зліва від рядка і за замовчуванням **вимкнений**.

> **Код у комірці нижче змінювати не потрібно.** Рядки `userdata.get('KAGGLE_USERNAME')` і `userdata.get('KAGGLE_KEY')` уже правильні: у дужках стоїть **назва** секрету, за якою Colab піде в панель і забере значення. Якщо вписати туди свій нікнейм або ключ, отримаєте `SecretNotFoundError: Secret <ваш нікнейм> does not exist` — Colab шукатиме секрет із такою назвою і не знайде його.

> **Чому саме так, а не змінними в комірці.** Секрети прив'язані до вашого Google-акаунта, а не до сесії: записали один раз — працює до кінця обох семестрів. Головне ж у тому, що ключ не потрапляє у файл блокнота. Якби ви вписали його прямо в код, він поїхав би разом із `.ipynb` у Moodle і в репозиторій GitHub — а GitHub боти сканують на такі ключі цілодобово.

> Це ваш особистий ключ. Не публікуйте його і не надсилайте нікому — ні викладачу, ні в чат групи. Якщо ключ усе ж кудись потрапив, це не катастрофа: поверніться в *Settings* → *API Tokens*, натисніть **`Expire Legacy API Key`**, потім **`Create Legacy API Key`** і впишіть у секрет нове значення.

In [5]:
import subprocess
import sys

install_result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "kaggle"],
    capture_output=True, text=True
)
if install_result.returncode == 0:
    print("Kaggle CLI готовий до використання.")
else:
    print("Не вдалося встановити Kaggle CLI:")
    print(install_result.stderr[-2000:])
    print("Перевірте підключення до Інтернету та повторіть цю комірку.")

Kaggle CLI готовий до використання.


In [6]:
import os
import shutil

KAGGLE_READY = False
kaggle_cmd = shutil.which("kaggle")

try:
    from google.colab import userdata

    kaggle_username = userdata.get("KAGGLE_USERNAME")
    kaggle_key = userdata.get("KAGGLE_KEY")

    if not kaggle_username or not kaggle_key:
        raise ValueError("Один або обидва секрети порожні.")

    os.environ["KAGGLE_USERNAME"] = kaggle_username
    os.environ["KAGGLE_KEY"] = kaggle_key

    if kaggle_cmd is None:
        raise RuntimeError("Команду kaggle не знайдено. Повторіть комірку встановлення Kaggle CLI.")

    KAGGLE_READY = True
    print("Секрети Kaggle отримано. Значення ключа не виводяться.")
except Exception as error:
    print("Kaggle API не налаштований:", error)
    print("У Colab додайте секрети KAGGLE_USERNAME і KAGGLE_KEY та ввімкніть Notebook access.")
    print("Залежні комірки Kaggle безпечно пропустять мережеві операції.")

Секрети Kaggle отримано. Значення ключа не виводяться.


In [7]:
import subprocess

if KAGGLE_READY:
    result = subprocess.run(
        [kaggle_cmd, "datasets", "list", "-s", "books", "--max-size", "52428800"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(result.stdout[:6000])
        print("Підключення до Kaggle API працює.")
    else:
        print("Kaggle API повернув помилку:")
        print(result.stderr[-2000:])
else:
    print("Перевірку Kaggle пропущено: спочатку налаштуйте секрети та Kaggle CLI.")

ref                                                               title                                                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
----------------------------------------------------------------  --------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
jealousleopard/goodreadsbooks                                     Goodreads-books                                         637338  2020-03-09 09:18:31.583000          93228       2027                1  
abdallahwagih/books-dataset                                       Books Dataset                                          1657228  2023-12-13 02:26:07.263000           8121        103                1  
dylanjcastillo/7k-books-with-metadata                             7k Books                                               1542454  2020-02-04 20:17:23.623000          19945         92          

### Завдання 2.3

Знайдіть на Kaggle набір даних, який вам цікавий, завантажте його і прочитайте у `pandas`.

Повторіть пошук за власним ключовим словом — замініть `books` на будь-яку тему, що вас цікавить.

> **Частина наборів не завантажиться, і це нормально.** Якщо у відповідь прийде `403 Client Error: Forbidden` — набір вимагає прийняти умови на його сторінці в браузері. Або прийміть їх, або просто візьміть наступний набір зі списку. Це не помилка у вашому коді.

In [8]:
import subprocess

if KAGGLE_READY:
    result = subprocess.run(
        [kaggle_cmd, "datasets", "list", "-s", "iris", "--max-size", "52428800"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(result.stdout[:6000])
    else:
        print("Пошук на Kaggle не виконано:")
        print(result.stderr[-2000:])
else:
    print("Пошук пропущено: Kaggle API не налаштований.")

ref                                                     title                                                size  lastUpdated                 downloadCount  voteCount  usabilityRating  
------------------------------------------------------  ---------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
uciml/iris                                              Iris Species                                         3687  2016-09-27 07:38:05.440000         962387       5005  0.7941176        
himanshunakrani/iris-dataset                            Iris dataset                                         1006  2022-07-20 18:50:06.277000         117327        453  1                
arshid/iris-flower-dataset                              Iris Flower Dataset                                  1010  2018-03-22 15:18:06.097000         260606       1160  0.8235294        
vikrishnan/iris-dataset                                 Iris Data

Для лабораторної обрано набір **`uciml/iris`** (Iris Dataset). Він містить невеликий CSV-файл із вимірюваннями квітів трьох видів ірисів та підходить для читання через Pandas.

Сторінка набору: https://www.kaggle.com/datasets/uciml/iris

Якщо Kaggle повертає `403 Forbidden`, відкрийте сторінку набору в браузері, увійдіть в акаунт і перевірте, чи потрібно прийняти умови використання.

In [9]:
from pathlib import Path
import subprocess

KAGGLE_DATA_DIR = Path("/content/kaggle_data")
KAGGLE_DATA_DIR.mkdir(parents=True, exist_ok=True)
KAGGLE_DOWNLOAD_OK = False

if KAGGLE_READY:
    result = subprocess.run(
        [kaggle_cmd, "datasets", "download", "-d", "uciml/iris",
         "-p", str(KAGGLE_DATA_DIR), "--unzip"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        KAGGLE_DOWNLOAD_OK = True
        print(result.stdout[-3000:] if result.stdout else "Набір Iris завантажено.")
        print("Файли у теці:")
        for file_path in sorted(KAGGLE_DATA_DIR.iterdir()):
            print(f"  {file_path.name} ({file_path.stat().st_size:,} байт)")
    else:
        print("Не вдалося завантажити набір із Kaggle:")
        print(result.stderr[-2500:])
        print("Перевірте Kaggle API, доступ до набору та повторіть комірку.")
else:
    print("Завантаження пропущено: Kaggle API не налаштований.")

Dataset URL: https://www.kaggle.com/datasets/uciml/iris
License(s): CC0-1.0


Файли у теці:
  Iris.csv (5,107 байт)
  database.sqlite (10,240 байт)


In [10]:
from pathlib import Path

csv_files = sorted(KAGGLE_DATA_DIR.glob("*.csv"))
kaggle_df = None

if csv_files:
    iris_path = next((p for p in csv_files if p.name.lower() == "iris.csv"), csv_files[0])
    try:
        kaggle_df = pd.read_csv(iris_path)
    except pd.errors.ParserError:
        # Автоматично визначаємо роздільник, якщо він відрізняється від коми.
        kaggle_df = pd.read_csv(iris_path, sep=None, engine="python")

    print("Файл:", iris_path.name)
    print("Розмір таблиці (рядки, стовпці):", kaggle_df.shape)
    print("Стовпці:", list(kaggle_df.columns))
    display(kaggle_df.head())
else:
    print("CSV-файл не знайдено. Завантаження Kaggle було пропущене або завершилося помилкою.")
    print("Після налаштування Kaggle API повторіть комірки 11, 16 і 17.")

Файл: Iris.csv
Розмір таблиці (рядки, стовпці): (150, 6)
Стовпці: ['Id', 'SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm', 'Species']


,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


## Крок 3. Google BigQuery Sandbox

BigQuery — хмарне аналітичне сховище, з яким ми працюватимемо в темах 4 і 5. Режим **Sandbox** дає безкоштовний доступ без прив'язки платіжної картки, з місячним лімітом обробки запитів у 1 ТБ. Для всіх завдань курсу цього вистачає з великим запасом.

**Підготовка.** Відкрийте console.cloud.google.com, створіть проєкт (назва довільна, наприклад `de-kpnu-2026`) і скопіюйте його **Project ID**.

> **Name і Project ID — різні речі.** Name — те, що ви вписали самі; Project ID — те, що згенерував Google, і воно може мати числовий хвіст (`de-kpnu-2026-472118`). Потрібен саме **ID**: він видно в колонці «ID» у селекторі проєктів або в адресному рядку після `project=`.

> **Зайдіть один раз у BigQuery через консоль** (меню ☰ → BigQuery). Перше відкриття активує Sandbox і вмикає BigQuery API для проєкту. Без цього з Colab прилетить `403 BigQuery API has not been used in project … or it is disabled`.

> **Якщо комірка автентифікації завершиться помилкою** `MessageError: Error: credential propagation was unsuccessful` — доступ ви надали, просто блокнот не встиг його підхопити. Виконайте цю саму комірку ще раз.

In [11]:
BQ_AUTHENTICATED = False

try:
    from google.colab import auth
    auth.authenticate_user()
    BQ_AUTHENTICATED = True
    print("Автентифікацію Google виконано.")
except Exception as error:
    print("Google-автентифікація не виконана:", error)
    print("BigQuery-комірки пропустять запити. Запустіть блокнот у Colab і повторіть цю комірку після входу в Google.")

Автентифікацію Google виконано.


In [13]:
import subprocess
import sys

def встановити_пакети(пакети):
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *пакети],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print("Помилка встановлення:", result.stderr[-1500:])
    return result.returncode == 0

try:
    from google.cloud import bigquery
except ImportError:
    if встановити_пакети(["google-cloud-bigquery", "db-dtypes"]):
        try:
            from google.cloud import bigquery
        except ImportError:
            bigquery = None
    else:
        bigquery = None

try:
    import db_dtypes
except ImportError:
    if встановити_пакети(["db-dtypes"]):
        try:
            import db_dtypes
        except ImportError:
            print("Попередження: db-dtypes не вдалося імпортувати.")

PROJECT_ID = "artmartkn1b23"

client = None
BQ_ENABLED = False
BQ_TABLE_READY = False
розмір_усі = None
розмір_два = None
bq_df = None
sql_two = None
коефіцієнт = None

if bigquery is None:
    print("BigQuery недоступний: бібліотеку не вдалося імпортувати.")
elif not BQ_AUTHENTICATED:
    print("BigQuery пропущено: спочатку виконайте Google-автентифікацію.")
elif not PROJECT_ID.strip():
    print("Вкажіть власний Project ID у PROJECT_ID, щоб увімкнути BigQuery.")
else:
    try:
        client = bigquery.Client(project=PROJECT_ID.strip())
        BQ_ENABLED = True
        print("Клієнт BigQuery створено для проєкту:", client.project)
    except Exception as error:
        print("Не вдалося створити клієнт BigQuery:", error)
        print("Перевірте Project ID, доступ до проєкту та активацію BigQuery API.")

Клієнт BigQuery створено для проєкту: artmartkn1b23


### Завдання 3.1

Працюємо з публічним набором `bigquery-public-data.usa_names` — це імена новонароджених у США за роками.

Подивіться, які в наборі є таблиці, і оберіть одну з них.

> У комірці нижче вже стоїть робоче значення `usa_1910_2013` — блокнот виконується від початку до кінця без помилок. Замініть його на будь-яку іншу таблицю зі списку, якщо хочете.

In [14]:
dataset_ref = "bigquery-public-data.usa_names"
tables = []
BQ_TABLE_READY = False

if BQ_ENABLED:
    try:
        tables = list(client.list_tables(dataset_ref))
        print(f"Набір містить {len(tables)} таблиць:")
        for table in tables:
            print("  ", table.table_id)
    except Exception as error:
        print("Не вдалося отримати список таблиць:", error)
        print("Перевірте BigQuery API та права доступу.")
else:
    print("Перегляд таблиць пропущено: BigQuery не підключений.")

Набір містить 2 таблиць:
   usa_1910_2013
   usa_1910_current


In [15]:
ТАБЛИЦЯ = "usa_1910_2013"
table_info = None

if BQ_ENABLED:
    try:
        table_info = client.get_table(f"{dataset_ref}.{ТАБЛИЦЯ}")
        BQ_TABLE_READY = True
        print("Назва таблиці:", table_info.full_table_id)
        print("Кількість рядків:", f"{table_info.num_rows:,}")
        print("Обсяг таблиці, МБ:", round(table_info.num_bytes / 1024**2, 2))
        print("Стовпці:")
        for field in table_info.schema:
            print(f"  {field.name}: {field.field_type}")
    except Exception as error:
        print("Не вдалося отримати метадані таблиці:", error)
        print("Перевірте Project ID, доступ і наявність таблиці.")
else:
    print("Метадані таблиці пропущено: BigQuery не підключений.")

Назва таблиці: bigquery-public-data:usa_names.usa_1910_2013
Кількість рядків: 5,552,452
Обсяг таблиці, МБ: 163.49
Стовпці:
  state: STRING
  gender: STRING
  year: INTEGER
  name: STRING
  number: INTEGER


### Завдання 3.2. Оцінка вартості запиту

У BigQuery ви платите за обсяг **просканованих** даних, а не за кількість повернутих рядків. Тому `LIMIT 10` не робить запит дешевим: якщо запит читає всю таблицю, ви заплатите за всю таблицю.

Оцінити обсяг можна заздалегідь, не виконуючи запит, — це **сухий прогін** (dry run). Звикайте робити його перед кожним новим запитом.

> **Якого числа очікувати.** Не десятикратної різниці. Економія приблизно дорівнює частці обраних стовпців у загальному обсязі таблиці: якщо ви взяли два «важких» стовпці з п'яти, різниця вийде близько двох разів — і це правильний результат, а не ознака помилки. Щоб побачити більший контраст, оцініть додатково запит до одного числового стовпця.

In [17]:
def оцінити(sql):
    """Оцінює обсяг даних для сканування через BigQuery dry run."""
    if not BQ_ENABLED:
        print("Dry run пропущено: BigQuery не підключений.")
        return None
    try:
        config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
        job = client.query(sql, job_config=config)
        bytes_processed = job.total_bytes_processed
        if bytes_processed is None:
            print("BigQuery не повернув оцінку обсягу сканування.")
            return None
        print(f"Просканує {bytes_processed / 1024**2:,.3f} МБ ({bytes_processed:,} байт)")
        return bytes_processed
    except Exception as error:
        print("Не вдалося оцінити SQL-запит:", error)
        return None

if BQ_TABLE_READY:
    sql_all = f"SELECT * FROM `{dataset_ref}.{ТАБЛИЦЯ}` LIMIT 10"
    розмір_усі = оцінити(sql_all)
else:
    print("Оцінку SELECT * пропущено: таблиця BigQuery недоступна.")

Просканує 163.491 МБ (171,432,506 байт)


In [18]:
if BQ_TABLE_READY:
    sql_two = f"""
    SELECT state, name
    FROM `{dataset_ref}.{ТАБЛИЦЯ}`
    LIMIT 10
    """
    розмір_два = оцінити(sql_two)
else:
    print("Оцінку запиту з двома стовпцями пропущено: BigQuery недоступний.")

Просканує 62.881 МБ (65,935,918 байт)


In [19]:
if BQ_TABLE_READY and sql_two is not None:
    try:
        bq_df = client.query(sql_two).to_dataframe()
        print("Розмір результату:", bq_df.shape)
        display(bq_df)
    except Exception as error:
        print("Не вдалося виконати SQL-запит:", error)
        print("Перевірте доступ до BigQuery та наявність пакета db-dtypes.")
else:
    print("SQL-запит пропущено: BigQuery не підключений.")

Розмір результату: (10, 2)


,state,name
0,AL,Vera
1,AR,Viola
2,AR,Alice
3,AR,Edna
4,AR,Ollie
5,CA,Beatrice
6,CT,Marion
7,CT,Marie
8,FL,Alice
9,GA,Thelma


In [20]:

if розмір_усі is not None and розмір_два is not None:
    if розмір_два > 0:
        коефіцієнт = розмір_усі / розмір_два
        print(f"SELECT * сканує приблизно у {коефіцієнт:.2f} раз(и) більше даних.")
    elif розмір_усі == 0 and розмір_два == 0:
        print("Обидва запити мають нульовий обсяг сканування; коефіцієнт не визначається.")
    else:
        print("Другий запит має нульовий обсяг сканування; коефіцієнт не визначається.")
else:
    print("Порівняння пропущено: немає оцінок обох запитів.")

SELECT * сканує приблизно у 2.60 раз(и) більше даних.


> **Результат завдання 3.2.** Фактичний коефіцієнт відображається у попередній комірці після успішного підключення до BigQuery. Якщо сервіс не був налаштований, заповніть Project ID, пройдіть автентифікацію та повторно виконайте комірки 19–28.

## Крок 4. GitHub

Матеріали курсу зберігаються в репозиторії, а ваші роботи протягом семестру складатимуться у власний. Для фінального проєкту репозиторій обовʼязковий.

**Створіть репозиторій** з іменем `data-engineering-2026` на github.com (приватний або публічний — на ваш розсуд).

### Завдання 4.1

1. Увійдіть у GitHub.
2. Натисніть **New repository**.
3. У полі Repository name введіть `data-engineering-2026`.
4. Оберіть Public або Private.
5. Натисніть **Create repository**.
6. У Google Colab відкрийте **Файл → Зберегти копію на GitHub**.
7. Виберіть створений репозиторій і збережіть блокнот.
8. Як повідомлення коміту вкажіть: `ЛР 1: налаштування середовища`.

**Увага:** перед публікацією перевірте, що в блокноті немає API-ключів. У цій роботі ключ Kaggle зберігається в секретах Colab, а не в коді.

## Висновки

Під час виконання лабораторної роботи було підготовлено робоче середовище для подальшого вивчення інженерії даних з Python. Створено робочу теку та перевірено запис і зчитування CSV-файлу за допомогою бібліотеки Pandas. Опрацьовано налаштування доступу до Kaggle API через секрети Google Colab і завантаження набору даних Iris. Також розглянуто порядок підключення до Google BigQuery Sandbox, перегляду структури публічної таблиці та порівняння обсягу сканування SQL-запитів. На завершення створено інструкцію зі збереження лабораторної роботи в репозиторії GitHub.

**Примітка:** операції з Google Drive, Kaggle, BigQuery та GitHub потребують авторизації у відповідних акаунтах. Якщо сервіс не налаштований, блокнот виводить пояснення та пропускає залежні мережеві операції без аварійного завершення. Після налаштування сервісу потрібно повторно виконати відповідні комірки, щоб отримати фактичні результати.

_(За потреби вкажіть фактичний коефіцієнт зменшення обсягу сканування, отриманий у завданні 3.2.)_